# Qwen3.5-4B joint release follow-up

Focused CUDA/Colab confirmation for the diagnostic winner from the whole-system matrix.

## Goal

Resolve the unmatched cache gate without repeating the search matrix, then test block-and-scale recovery on the actual winner: uniform W4 plus the frozen mixed 3.25-bpv Gaussian K/V map.

The notebook runs four required new trials and at most four conditional confirmation trials. It deliberately omits dynamic allocation and the LoRA-QAT profile that previously restored step 0.

### Key assumptions and predeclared gates

- Baseline artifacts come from Git SHA `8d9676f109fa`; executable experiment code must be unchanged at the checked-out revision.
- The frozen recipe, candidate seeds 0/1/2, and seed-0 K4/V4 control are reused from that completed run.
- Cache quality is compared within the same weight recipe and seed: candidate KL divided by matched K4/V4-control KL.
- Release gates: mean PPL degradation at most 5%, worst-seed PPL degradation at most 10%, and worst matched cache-KL ratio at most 1.05.
- Recovery is promoted from seed 0 only if it improves PPL by at least 0.2%, preserves at least 56.5% estimated weight reduction, passes its matched cache gate, and retains no adapter.
- CUDA fallback results are quality measurements plus logical byte estimates; they are not VRAM or throughput measurements.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-joint-release-followup")
MODEL_ID = "unsloth/Qwen3.5-4B"
CONFIG_RELATIVE_PATH = Path("configs/qwen35_4b_joint_cuda.yaml")

USE_GOOGLE_DRIVE = True
DRIVE_JOINT_RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_joint_matrix")
DRIVE_FOLLOWUP_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_joint_release_followup")
DRIVE_KV_RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_kv_matrix")
LOCAL_RESULT_ROOT = Path("/content/qwen35_joint_release_followup")
BASELINE_RUN = "8d9676f109fa"
FROZEN_KV_TRANSFER_RUN = "49d3cf1182f0"

EVAL_SEQ_LEN = 256
EVAL_MAX_SAMPLES = 64
MEAN_PPL_GATE = 0.05
WORST_PPL_GATE = 0.10
MATCHED_CACHE_KL_RATIO_GATE = 1.05
RECOVERY_PPL_MIN_RELATIVE_IMPROVEMENT = 0.002
MIN_WEIGHT_REDUCTION = 0.565
PPL_MATCH_TOLERANCE = 1e-4

CONFIRM_EXPENSIVE_RUN = False
FORCE_RERUN = False
RUN_RECOVERY_CONFIRMATION_IF_PROMOTED = True
FORCE_RECOVERY_CONFIRMATION = False
DOWNLOAD_RESULTS = True

print({
    "repo_ref": REPO_REF,
    "baseline_run": BASELINE_RUN,
    "required_new_runs": 4,
    "conditional_new_runs": 4,
    "confirm_expensive_run": CONFIRM_EXPENSIVE_RUN,
})

### 1. Verify the CUDA runtime

In [ ]:
import os
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB")
print(f"torch={torch.__version__} | CUDA={torch.version.cuda}")
if vram_gib < 40:
    print("WARNING: cached fallback may OOM below 40 GiB.")
subprocess.run(["nvidia-smi"], check=True)

### 2. Mount Drive and fetch the latest revision

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_BASE = DRIVE_FOLLOWUP_ROOT
    BASELINE_RESULT_ROOT = DRIVE_JOINT_RESULT_ROOT / BASELINE_RUN
    FROZEN_KV_SUMMARY_PATH = DRIVE_KV_RESULT_ROOT / "frozen_transfer" / FROZEN_KV_TRANSFER_RUN / "kv_frozen_transfer_summary.json"
else:
    RESULT_BASE = LOCAL_RESULT_ROOT
    BASELINE_RESULT_ROOT = LOCAL_RESULT_ROOT / "baseline"
    FROZEN_KV_SUMMARY_PATH = LOCAL_RESULT_ROOT / "frozen_transfer_summary.json"
RESULT_BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_DIR, check=True)

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
RESULT_ROOT = RESULT_BASE / commit[:12]
RUN_ROOT = RESULT_ROOT / "runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using commit {commit}; results: {RESULT_ROOT}")

### 3. Install dependencies without replacing CUDA PyTorch

An `HF_TOKEN` Colab secret is optional. The public model downloads without one, but authenticated requests have higher rate limits.

In [ ]:
runtime_packages = [
    "transformers==5.9.0", "datasets>=4.8", "accelerate",
    "safetensors", "sentencepiece", "scipy", "pyyaml",
    "pandas", "matplotlib", "huggingface_hub",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)

pil_probe_command = [
    sys.executable, "-c",
    "from PIL import Image, ImageColor, ImageDraw, ImageFont, ImageText; print(Image.__version__)",
]
pil_probe = subprocess.run(pil_probe_command, capture_output=True, text=True, check=False)
if pil_probe.returncode != 0:
    print("Detected an inconsistent live Pillow installation; repairing it once.")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
        "--no-cache-dir", "pillow==12.2.0",
    ], check=True)
    subprocess.run(pil_probe_command, check=True)
    raise RuntimeError(
        "Pillow was repaired. Use Runtime > Restart session, then rerun from the top."
    )

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
print(f"Runtime ready; Pillow={pil_probe.stdout.strip()}.")

### 4. Validate the repository and baseline artifacts

In [ ]:
import json

import yaml

config_path = REPO_DIR / CONFIG_RELATIVE_PATH
baseline_summary_path = BASELINE_RESULT_ROOT / "joint_summary.json"
baseline_joint_path = BASELINE_RESULT_ROOT / "joint_matrix.csv"
baseline_release_path = BASELINE_RESULT_ROOT / "release_validation.csv"
required_paths = [
    config_path,
    REPO_DIR / "rotquant/dynamic.py",
    REPO_DIR / "rotquant/eval/kv_cache.py",
    REPO_DIR / "scripts/run_experiment.py",
    baseline_summary_path, baseline_joint_path, baseline_release_path,
    FROZEN_KV_SUMMARY_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, "Missing required files: " + ", ".join(missing)

with config_path.open() as handle:
    base_config = yaml.safe_load(handle)
with baseline_summary_path.open() as handle:
    baseline_summary = json.load(handle)
with FROZEN_KV_SUMMARY_PATH.open() as handle:
    frozen_kv_summary = json.load(handle)
baseline_git_sha = baseline_summary["git_sha"]
assert baseline_git_sha.startswith(BASELINE_RUN)
assert baseline_summary["winner"] == "uniform_w4__frozen_mixed_3.25"
assert frozen_kv_summary["recommendation"] == "mixed"
assert frozen_kv_summary["source_run_id"] == "bf94f2045a90"
FROZEN_MIXED_KV_RECIPE = frozen_kv_summary["mixed_recipe"]
assert FROZEN_MIXED_KV_RECIPE and all(
    {"layer", "key_bits", "value_bits"} <= set(row)
    for row in FROZEN_MIXED_KV_RECIPE
)

relevant_paths = [
    "rotquant", "eval", "scripts/run_experiment.py",
    str(CONFIG_RELATIVE_PATH),
]
code_drift = subprocess.run(
    ["git", "diff", "--quiet", baseline_git_sha, commit, "--", *relevant_paths],
    cwd=REPO_DIR,
    check=False,
)
assert code_drift.returncode == 0, (
    "Experiment or evaluation code changed since the baseline run. "
    "Do not mix old candidate rows with new controls; rebuild the comparison."
)
assert base_config["model"] == MODEL_ID
assert base_config["patch"]["fallback"] is True
print({
    "baseline_git_sha": baseline_git_sha,
    "current_git_sha": commit,
    "code_drift": False,
    "baseline_winner": baseline_summary["winner"],
})

## Steps

### 5. Define the two cache arms and block recovery

In [ ]:
def encoded_override(path, value):
    return f"{path}={json.dumps(value, separators=(',', ':'))}"

def uniform_weight(bits):
    return [
        "patch.enabled=true", f"quant.bits={bits}",
        encoded_override("patch.dynamic", None),
    ]

def uniform_kv(key_bits, value_bits, *, codebook="gaussian", group_size=64):
    return [
        encoded_override("eval.kv_cache.dynamic", None),
        encoded_override("eval.kv_cache.frozen_recipe", None),
        f"eval.kv_cache.key_bits={key_bits}",
        f"eval.kv_cache.value_bits={value_bits}",
        f"eval.kv_cache.codebook={codebook}",
        f"eval.kv_cache.group_size={group_size}",
    ]

def frozen_kv(recipe, *, codebook="gaussian"):
    return [
        "eval.kv_cache.key_bits=null",
        "eval.kv_cache.value_bits=null",
        f"eval.kv_cache.codebook={codebook}",
        encoded_override("eval.kv_cache.dynamic", None),
        encoded_override("eval.kv_cache.frozen_recipe", recipe),
    ]

COMMON_EVAL = [
    "eval.perplexity=true",
    f"eval.ppl.seq_len={EVAL_SEQ_LEN}",
    f"eval.ppl.max_samples={EVAL_MAX_SAMPLES}",
    "eval.kv_cache.batches=4",
    "eval.kv_cache.eval_offset_batches=4",
]
UNIFORM_W4 = uniform_weight(4)
FROZEN_MIXED_KV = frozen_kv(FROZEN_MIXED_KV_RECIPE)
UNIFORM_K4_V4 = uniform_kv(4, 4)

block_settings = {
    "objective": "block", "steps": 12, "lr": 0.0015,
    "train_batches": 4, "validation_batches": 2, "selection_batches": 2,
    "learn_scales": True, "scale_lr": 0.01,
    "scale_multiplier_min": 0.5, "scale_multiplier_max": 1.5,
    "propagate_quantized_inputs": True, "max_grad_norm": 1.0,
    "restore_best": True, "early_stopping_patience": 4,
    "validation_min_improvement": 0.001,
    "selection_min_improvement": 0.005,
    "distill_steps": 0,
}
BLOCK_RECOVERY = [
    "patch.rotation=butterfly",
    encoded_override("patch.train_rotation", block_settings),
]
print({
    "frozen_recipe": FROZEN_MIXED_KV_RECIPE,
    "block_recovery": block_settings,
})

### 6. Define the content-addressed experiment runner

In [ ]:
import hashlib
import shlex
from collections.abc import Iterable

PROTOCOL_VERSION = "joint-release-followup-v1"
trial_records = {}
trial_registry = {}
registry_path = RESULT_ROOT / "trial_registry.json"

def latest_result(output_dir):
    candidates = sorted(output_dir.glob("*.json"), key=lambda path: path.stat().st_mtime)
    if not candidates:
        return None
    with candidates[-1].open() as handle:
        return json.load(handle)

def persist_registry():
    with registry_path.open("w") as handle:
        json.dump(list(trial_registry.values()), handle, indent=2)

def run_logged(command, *, cwd, log_path):
    with log_path.open("w") as log_handle:
        process = subprocess.Popen(
            command, cwd=cwd, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            log_handle.flush()
        returncode = process.wait()
    if returncode != 0:
        lines = log_path.read_text(errors="replace").splitlines()
        tail = "\n".join(lines[-120:])
        raise RuntimeError(
            f"Trial subprocess failed with exit code {returncode}. "
            f"Full log: {log_path}\n--- log tail ---\n{tail}"
        )

def run_trial(stage, trial_name, overrides: Iterable[str], *, seed, weight, kv):
    overrides = list(overrides)
    signature = hashlib.sha256(json.dumps({
        "protocol": PROTOCOL_VERSION,
        "commit": commit,
        "seed": seed,
        "overrides": overrides,
    }, sort_keys=True).encode()).hexdigest()[:12]
    output_dir = RUN_ROOT / signature
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = None if FORCE_RERUN else latest_result(output_dir)
    if payload is None:
        command = [
            sys.executable, str(REPO_DIR / "scripts/run_experiment.py"),
            str(config_path), "--output-dir", str(output_dir),
            "--seed", str(seed),
        ]
        for override in overrides:
            command.extend(["--set", override])
        print("Running:", shlex.join(command), flush=True)
        run_logged(command, cwd=REPO_DIR, log_path=output_dir / "subprocess.log")
        payload = latest_result(output_dir)
        assert payload is not None, f"No result JSON was written to {output_dir}"
    else:
        print(f"Reusing {trial_name}: {signature}")

    key = f"{stage}:{trial_name}:s{seed}"
    trial_records[key] = payload
    trial_registry[key] = {
        "key": key, "stage": stage, "trial": trial_name,
        "weight": weight, "kv": kv, "seed": seed,
        "signature": signature, "overrides": overrides,
        "result_dir": str(output_dir),
    }
    persist_registry()
    return payload

### 7. Define comparable reporting rows

In [ ]:
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

source_ppl = float(baseline_summary["source_ppl"])
index_path = hf_hub_download(MODEL_ID, "model.safetensors.index.json")
with open(index_path) as handle:
    source_weight_bytes = json.load(handle)["metadata"]["total_size"]

def estimated_complete_weight_bytes(payload):
    metrics = payload["metrics"]
    if "fp16_weight_bytes" not in metrics:
        return source_weight_bytes
    deployed = metrics.get(
        "packed_plus_auxiliary_bytes", metrics["packed_weight_bytes"]
    )
    return source_weight_bytes - metrics["fp16_weight_bytes"] + deployed

def report_row(stage, trial, payload, *, weight, kv, seed):
    metrics = payload["metrics"]
    cache = metrics.get("kv_cache", {})
    trajectory = metrics.get("trajectory", {})
    weight_bytes = estimated_complete_weight_bytes(payload)
    cache_bytes = float(cache.get("deployed_total_cache_bytes", 0.0))
    ppl = float(metrics.get("ppl_wikitext2", np.nan))
    return {
        "stage": stage, "trial": trial, "weight": weight, "kv": kv, "seed": seed,
        "ppl": ppl,
        "relative_ppl": ppl / source_ppl - 1.0 if np.isfinite(ppl) else np.nan,
        "weight_bpw": metrics.get("effective_bits_per_weight", metrics.get("bits_per_weight_mean", 16.0)),
        "estimated_weight_GB": weight_bytes / 1e9,
        "weight_reduction": 1.0 - weight_bytes / source_weight_bytes,
        "rotation_MB": metrics.get("rotation_parameter_bytes", 0) / 1e6,
        "adapter_MB": metrics.get("adapter_parameter_bytes", 0) / 1e6,
        "cache_kl": cache.get("mean_teacher_kl", np.nan),
        "cache_nll_delta": cache.get("nll_delta", np.nan),
        "cache_top1": cache.get("top1_agreement", np.nan),
        "cache_cosine": cache.get("mean_logit_cosine", np.nan),
        "effective_kv_bpv": cache.get("effective_kv_bpv", np.nan),
        "prefill_key_nmse": cache.get("prefill_key_nmse", np.nan),
        "prefill_value_nmse": cache.get("prefill_value_nmse", np.nan),
        "deployed_cache_MB": cache_bytes / 1e6,
        "total_system_GB_at_prompt": (weight_bytes + cache_bytes) / 1e9,
        "trajectory_token_agreement": trajectory.get("token_agreement", np.nan),
        "trajectory_exact_rate": trajectory.get("exact_trajectory_rate", np.nan),
    }

def validation_summary(frame):
    return {
        "mean_ppl": float(frame["ppl"].mean()),
        "mean_relative_ppl": float(frame["ppl"].mean() / source_ppl - 1.0),
        "worst_relative_ppl": float(frame["relative_ppl"].max()),
        "mean_cache_kl_ratio": float(frame["cache_kl_ratio_to_k4v4"].mean()),
        "worst_cache_kl_ratio": float(frame["cache_kl_ratio_to_k4v4"].max()),
    }

def passes_release_gates(summary):
    return bool(
        summary["mean_relative_ppl"] <= MEAN_PPL_GATE
        and summary["worst_relative_ppl"] <= WORST_PPL_GATE
        and summary["worst_cache_kl_ratio"] <= MATCHED_CACHE_KL_RATIO_GATE
    )

### 8. Confirm the focused run

Set `CONFIRM_EXPENSIVE_RUN = True` only after reviewing the parameters above.

In [ ]:
assert CONFIRM_EXPENSIVE_RUN, (
    "Review the gates and run count, then set CONFIRM_EXPENSIVE_RUN=True."
)
print("Confirmed: four required trials, with four more only if seed-0 recovery is promoted.")

### 9. Load the completed winner and seed-0 control

In [ ]:
baseline_joint = pd.read_csv(baseline_joint_path)
baseline_release = pd.read_csv(baseline_release_path)
BASE_TRIAL = "uniform_w4__frozen_mixed_3.25"
CONTROL_TRIAL = "uniform_w4__uniform_k4_v4"
baseline_candidate = baseline_release[
    baseline_release["trial"] == BASE_TRIAL
].copy().sort_values("seed")
baseline_control_seed0 = baseline_joint[
    baseline_joint["trial"] == CONTROL_TRIAL
].copy()
assert baseline_candidate["seed"].tolist() == [0, 1, 2]
assert len(baseline_control_seed0) == 1
assert int(baseline_control_seed0.iloc[0]["seed"]) == 0
assert np.allclose(
    baseline_candidate["relative_ppl"],
    baseline_candidate["ppl"] / source_ppl - 1.0,
)
display(baseline_candidate[[
    "trial", "seed", "ppl", "relative_ppl", "cache_kl",
    "effective_kv_bpv", "estimated_weight_GB", "weight_reduction",
]])
display(baseline_control_seed0[[
    "trial", "seed", "ppl", "cache_kl", "effective_kv_bpv",
]])

### 10. Run the missing K4/V4 controls for seeds 1 and 2

These two controls resolve the original seed-0-to-worst-seed comparison error.

In [ ]:
unrecovered_control_rows = [baseline_control_seed0.iloc[0].to_dict()]
for seed in (1, 2):
    payload = run_trial(
        "matched_control", CONTROL_TRIAL,
        [*UNIFORM_W4, *UNIFORM_K4_V4, *COMMON_EVAL],
        seed=seed, weight="uniform_w4", kv="uniform_k4_v4",
    )
    unrecovered_control_rows.append(report_row(
        "matched_control", CONTROL_TRIAL, payload,
        weight="uniform_w4", kv="uniform_k4_v4", seed=seed,
    ))
unrecovered_controls = pd.DataFrame(unrecovered_control_rows).sort_values("seed")
assert unrecovered_controls["seed"].astype(int).tolist() == [0, 1, 2]

unrecovered_matched = baseline_candidate.merge(
    unrecovered_controls[["seed", "cache_kl"]].rename(
        columns={"cache_kl": "control_cache_kl"}
    ),
    on="seed", validate="one_to_one",
)
unrecovered_matched["cache_kl_ratio_to_k4v4"] = (
    unrecovered_matched["cache_kl"] / unrecovered_matched["control_cache_kl"]
)
unrecovered_summary = validation_summary(unrecovered_matched)
unrecovered_release_pass = passes_release_gates(unrecovered_summary)
print({**unrecovered_summary, "release_gate_pass": unrecovered_release_pass})
display(unrecovered_matched[[
    "seed", "ppl", "relative_ppl", "cache_kl",
    "control_cache_kl", "cache_kl_ratio_to_k4v4",
]].style.format({
    "ppl": "{:.4f}", "relative_ppl": "{:+.2%}",
    "cache_kl": "{:.6f}", "control_cache_kl": "{:.6f}",
    "cache_kl_ratio_to_k4v4": "{:.3f}x",
}))

### 11. Test block recovery on the actual winner at seed 0

Both arms repeat the same deterministic weight recovery and differ only in the K/V recipe. Their PPL and weight footprint must match.

In [ ]:
RECOVERED_TRIAL = "uniform_w4_block_scale__frozen_mixed_3.25"
RECOVERED_CONTROL_TRIAL = "uniform_w4_block_scale__uniform_k4_v4"
recovered_frozen_overrides = [
    *UNIFORM_W4, *BLOCK_RECOVERY, *FROZEN_MIXED_KV, *COMMON_EVAL,
]
recovered_control_overrides = [
    *UNIFORM_W4, *BLOCK_RECOVERY, *UNIFORM_K4_V4, *COMMON_EVAL,
]
seed0_recovered_payload = run_trial(
    "recovery", RECOVERED_TRIAL, recovered_frozen_overrides,
    seed=0, weight="uniform_w4+block_scale", kv="frozen_mixed_3.25",
)
seed0_recovered_control_payload = run_trial(
    "recovery_control", RECOVERED_CONTROL_TRIAL, recovered_control_overrides,
    seed=0, weight="uniform_w4+block_scale", kv="uniform_k4_v4",
)
seed0_recovered_row = report_row(
    "recovery", RECOVERED_TRIAL, seed0_recovered_payload,
    weight="uniform_w4+block_scale", kv="frozen_mixed_3.25", seed=0,
)
seed0_recovered_control_row = report_row(
    "recovery_control", RECOVERED_CONTROL_TRIAL, seed0_recovered_control_payload,
    weight="uniform_w4+block_scale", kv="uniform_k4_v4", seed=0,
)
base_seed0_ppl = float(baseline_candidate.loc[baseline_candidate["seed"] == 0, "ppl"].iloc[0])
seed0_ppl_improvement = 1.0 - seed0_recovered_row["ppl"] / base_seed0_ppl
seed0_cache_ratio = (
    seed0_recovered_row["cache_kl"] / seed0_recovered_control_row["cache_kl"]
)
seed0_ppl_matches = abs(
    seed0_recovered_row["ppl"] - seed0_recovered_control_row["ppl"]
) <= PPL_MATCH_TOLERANCE
seed0_recovery_checks = {
    "ppl_improvement": seed0_ppl_improvement,
    "ppl_improvement_pass": seed0_ppl_improvement >= RECOVERY_PPL_MIN_RELATIVE_IMPROVEMENT,
    "cache_kl_ratio_to_k4v4": seed0_cache_ratio,
    "cache_gate_pass": seed0_cache_ratio <= MATCHED_CACHE_KL_RATIO_GATE,
    "weight_reduction": seed0_recovered_row["weight_reduction"],
    "weight_gate_pass": seed0_recovered_row["weight_reduction"] >= MIN_WEIGHT_REDUCTION,
    "adapter_MB": seed0_recovered_row["adapter_MB"],
    "adapter_gate_pass": seed0_recovered_row["adapter_MB"] == 0.0,
    "matched_weight_ppl": seed0_ppl_matches,
}
seed0_recovery_promoted = bool(
    seed0_recovery_checks["ppl_improvement_pass"]
    and seed0_recovery_checks["cache_gate_pass"]
    and seed0_recovery_checks["weight_gate_pass"]
    and seed0_recovery_checks["adapter_gate_pass"]
    and seed0_recovery_checks["matched_weight_ppl"]
)
print({**seed0_recovery_checks, "promote_to_seed_confirmation": seed0_recovery_promoted})
display(pd.DataFrame([seed0_recovered_row, seed0_recovered_control_row]))

### 12. Conditionally validate recovered weights at seeds 1 and 2

The four confirmation trials run only when seed-0 recovery passes every promotion check, unless `FORCE_RECOVERY_CONFIRMATION=True`.

In [ ]:
recovered_candidate_rows = [seed0_recovered_row]
recovered_control_rows = [seed0_recovered_control_row]
run_recovery_confirmation = bool(
    FORCE_RECOVERY_CONFIRMATION
    or (RUN_RECOVERY_CONFIRMATION_IF_PROMOTED and seed0_recovery_promoted)
)
if run_recovery_confirmation:
    for seed in (1, 2):
        candidate_payload = run_trial(
            "recovery_validation", RECOVERED_TRIAL, recovered_frozen_overrides,
            seed=seed, weight="uniform_w4+block_scale", kv="frozen_mixed_3.25",
        )
        control_payload = run_trial(
            "recovery_validation_control", RECOVERED_CONTROL_TRIAL, recovered_control_overrides,
            seed=seed, weight="uniform_w4+block_scale", kv="uniform_k4_v4",
        )
        recovered_candidate_rows.append(report_row(
            "recovery_validation", RECOVERED_TRIAL, candidate_payload,
            weight="uniform_w4+block_scale", kv="frozen_mixed_3.25", seed=seed,
        ))
        recovered_control_rows.append(report_row(
            "recovery_validation_control", RECOVERED_CONTROL_TRIAL, control_payload,
            weight="uniform_w4+block_scale", kv="uniform_k4_v4", seed=seed,
        ))
else:
    print("Seed-0 recovery was not promoted; skipping four confirmation trials.")

recovered_candidates = pd.DataFrame(recovered_candidate_rows).sort_values("seed")
recovered_controls = pd.DataFrame(recovered_control_rows).sort_values("seed")
recovered_matched = recovered_candidates.merge(
    recovered_controls[["seed", "cache_kl", "ppl"]].rename(columns={
        "cache_kl": "control_cache_kl",
        "ppl": "control_ppl",
    }),
    on="seed", validate="one_to_one",
)
recovered_matched["cache_kl_ratio_to_k4v4"] = (
    recovered_matched["cache_kl"] / recovered_matched["control_cache_kl"]
)
recovered_matched["matched_weight_ppl"] = (
    (recovered_matched["ppl"] - recovered_matched["control_ppl"]).abs()
    <= PPL_MATCH_TOLERANCE
)
display(recovered_matched[[
    "seed", "ppl", "control_ppl", "matched_weight_ppl",
    "relative_ppl", "cache_kl", "control_cache_kl",
    "cache_kl_ratio_to_k4v4", "weight_reduction",
]])

## Checks and results

### 13. Apply the final matched release gates

In [ ]:
recovered_summary = None
recovered_release_pass = False
recovered_mean_improvement = None
recovered_promoted = False
if len(recovered_matched) == 3:
    assert recovered_matched["matched_weight_ppl"].all()
    recovered_summary = validation_summary(recovered_matched)
    recovered_release_pass = passes_release_gates(recovered_summary)
    recovered_mean_improvement = (
        1.0 - recovered_summary["mean_ppl"] / unrecovered_summary["mean_ppl"]
    )
    recovered_promoted = bool(
        recovered_release_pass
        and recovered_mean_improvement >= RECOVERY_PPL_MIN_RELATIVE_IMPROVEMENT
        and recovered_matched["weight_reduction"].min() >= MIN_WEIGHT_REDUCTION
        and recovered_matched["adapter_MB"].max() == 0.0
    )

if recovered_promoted:
    winner = RECOVERED_TRIAL
    release_status = "matched_release_gates_passed"
elif unrecovered_release_pass:
    winner = BASE_TRIAL
    release_status = "matched_release_gates_passed"
else:
    winner = BASE_TRIAL
    release_status = "diagnostic_only"

comparison_rows = [{
    "trial": BASE_TRIAL,
    **unrecovered_summary,
    "release_gate_pass": unrecovered_release_pass,
    "mean_ppl_improvement_vs_base": 0.0,
}]
if recovered_summary is not None:
    comparison_rows.append({
        "trial": RECOVERED_TRIAL,
        **recovered_summary,
        "release_gate_pass": recovered_release_pass,
        "mean_ppl_improvement_vs_base": recovered_mean_improvement,
    })
comparison_table = pd.DataFrame(comparison_rows)
print({
    "winner": winner,
    "release_status": release_status,
    "recovered_promoted": recovered_promoted,
})
display(comparison_table.style.format({
    "mean_ppl": "{:.4f}",
    "mean_relative_ppl": "{:+.2%}",
    "worst_relative_ppl": "{:+.2%}",
    "mean_cache_kl_ratio": "{:.3f}x",
    "worst_cache_kl_ratio": "{:.3f}x",
    "mean_ppl_improvement_vs_base": "{:+.2%}",
}))

### 14. Plot PPL and matched cache ratios

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(
    unrecovered_matched["seed"], unrecovered_matched["ppl"],
    marker="o", label="uniform W4 + frozen mixed",
)
if len(recovered_matched) == 3:
    axes[0].plot(
        recovered_matched["seed"], recovered_matched["ppl"],
        marker="o", label="block recovered + frozen mixed",
    )
axes[0].axhline(source_ppl * (1 + MEAN_PPL_GATE), linestyle="--", color="tab:orange", label="5% mean gate")
axes[0].axhline(source_ppl * (1 + WORST_PPL_GATE), linestyle=":", color="tab:red", label="10% worst gate")
axes[0].set(title="WikiText-2 PPL by seed", xlabel="Seed", ylabel="PPL")
axes[0].set_xticks([0, 1, 2])
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].plot(
    unrecovered_matched["seed"], unrecovered_matched["cache_kl_ratio_to_k4v4"],
    marker="o", label="uniform W4 + frozen mixed",
)
if len(recovered_matched) == 3:
    axes[1].plot(
        recovered_matched["seed"], recovered_matched["cache_kl_ratio_to_k4v4"],
        marker="o", label="block recovered + frozen mixed",
    )
axes[1].axhline(MATCHED_CACHE_KL_RATIO_GATE, linestyle="--", color="tab:red", label="matched KL gate")
axes[1].axhline(1.0, linestyle=":", color="black", alpha=0.6, label="K4/V4 control")
axes[1].set(title="Candidate cache KL / matched K4/V4 KL", xlabel="Seed", ylabel="KL ratio (lower is better)")
axes[1].set_xticks([0, 1, 2])
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
plt.tight_layout()
comparison_plot_path = RESULT_ROOT / "matched_release_comparison.png"
fig.savefig(comparison_plot_path, dpi=170)
plt.show()
print(comparison_plot_path)

### 15. Validate protocol invariants

In [ ]:
assert unrecovered_matched["seed"].astype(int).tolist() == [0, 1, 2]
assert (unrecovered_matched["effective_kv_bpv"] == 3.25).all()
assert np.isfinite(unrecovered_matched["cache_kl_ratio_to_k4v4"]).all()
assert seed0_ppl_matches, "Recovered candidate/control PPL differs despite identical weights."
assert seed0_recovered_row["adapter_MB"] == 0.0
if len(recovered_matched) == 3:
    assert recovered_matched["seed"].astype(int).tolist() == [0, 1, 2]
    assert recovered_matched["matched_weight_ppl"].all()
    assert np.isfinite(recovered_matched["cache_kl_ratio_to_k4v4"]).all()

for key, payload in trial_records.items():
    cache_config = payload["config"].get("eval", {}).get("kv_cache")
    assert cache_config is not None, key
    assert int(cache_config.get("eval_offset_batches", 0)) >= 0
    cache_metrics = payload["metrics"]["kv_cache"]
    assert cache_metrics["prefill_key_nmse"] >= 0
    assert cache_metrics["prefill_value_nmse"] >= 0
    frozen_recipe = cache_config.get("frozen_recipe") or []
    if frozen_recipe:
        assert cache_metrics["frozen_recipe"]["validated_layers"] == len(frozen_recipe), key
print(f"Protocol checks passed for {len(trial_records)} new trial records.")

### 16. Persist the audit trail and compact archive

In [ ]:
import shutil

unrecovered_control_path = RESULT_ROOT / "matched_k4v4_controls.csv"
unrecovered_validation_path = RESULT_ROOT / "unrecovered_matched_validation.csv"
recovery_validation_path = RESULT_ROOT / "recovery_matched_validation.csv"
comparison_path = RESULT_ROOT / "release_comparison.csv"
unrecovered_controls.to_csv(unrecovered_control_path, index=False)
unrecovered_matched.to_csv(unrecovered_validation_path, index=False)
recovered_matched.to_csv(recovery_validation_path, index=False)
comparison_table.to_csv(comparison_path, index=False)

summary = {
    "git_sha": commit,
    "baseline_git_sha": baseline_git_sha,
    "model": MODEL_ID,
    "source_ppl": source_ppl,
    "matched_cache_kl_ratio_gate": MATCHED_CACHE_KL_RATIO_GATE,
    "unrecovered": {**unrecovered_summary, "release_gate_pass": unrecovered_release_pass},
    "seed0_recovery_checks": seed0_recovery_checks,
    "recovery_confirmation_ran": run_recovery_confirmation,
    "recovered": None if recovered_summary is None else {
        **recovered_summary,
        "release_gate_pass": recovered_release_pass,
        "mean_ppl_improvement_vs_base": recovered_mean_improvement,
        "promoted": recovered_promoted,
    },
    "winner": winner,
    "release_status": release_status,
    "frozen_kv_source": str(FROZEN_KV_SUMMARY_PATH),
    "baseline_artifacts": {
        "summary": str(baseline_summary_path),
        "joint_matrix": str(baseline_joint_path),
        "release_validation": str(baseline_release_path),
    },
    "fallback_memory_warning": "CUDA fallback VRAM is invalid; byte metrics are logical deployment estimates.",
}
summary_path = RESULT_ROOT / "joint_release_followup_summary.json"
with summary_path.open("w") as handle:
    json.dump(summary, handle, indent=2)

log_lines = [
    f"## Qwen3.5-4B joint release follow-up ({commit[:12]})",
    "",
    f"- Baseline run: {baseline_git_sha}",
    f"- Unrecovered matched release pass: {unrecovered_release_pass}",
    f"- Unrecovered mean/worst relative PPL: {unrecovered_summary['mean_relative_ppl']:+.2%} / {unrecovered_summary['worst_relative_ppl']:+.2%}",
    f"- Unrecovered mean/worst matched cache ratio: {unrecovered_summary['mean_cache_kl_ratio']:.4f} / {unrecovered_summary['worst_cache_kl_ratio']:.4f}",
    f"- Seed-0 recovery promoted: {seed0_recovery_promoted}",
    f"- Recovery confirmation ran: {run_recovery_confirmation}",
    f"- Winner: {winner}",
    f"- Release status: {release_status}",
    "- Candidate and K4/V4 cache KL were compared within the same weight recipe and seed.",
    "- CUDA fallback measurements are quality-only; packed byte accounting is logical.",
]
log_path = RESULT_ROOT / "experiment_log_entry.md"
log_path.write_text("\n".join(log_lines) + "\n")

archive_base = RESULT_BASE / f"qwen35_joint_release_followup_{commit[:12]}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", RESULT_ROOT))
print(json.dumps(summary, indent=2))
print({
    "matched_controls": str(unrecovered_control_path),
    "unrecovered_validation": str(unrecovered_validation_path),
    "recovery_validation": str(recovery_validation_path),
    "comparison": str(comparison_path),
    "summary": str(summary_path),
    "experiment_log_entry": str(log_path),
    "archive": str(archive_path),
})

### 17. Download the compact result archive

In [ ]:
if DOWNLOAD_RESULTS:
    if USE_GOOGLE_DRIVE:
        from google.colab import files
        files.download(str(archive_path))
    else:
        print(f"Archive ready: {archive_path}")
else:
    print(f"Results retained at {RESULT_ROOT}")

## Next steps

- Treat `matched_release_gates_passed` as a development-protocol result, not a deployment claim.
- If the unrecovered winner passes, export and validate the actual packed artifact before quoting memory reduction.
- If block recovery is promoted, carry only that recovered recipe into long-context perplexity and retrieval tests.
- Do not reintroduce LoRA-QAT until its step-0 collapse has a concrete fix.